In [ ]:
from jax import random
from jax import numpy as jnp
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import numpyro
from numpyro import distributions as dist
from numpyro import infer
import arviz as az
from plotly.express.colors import qualitative as qual_colours
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

from estival.sampling import tools as esamp

from emu_renewal.inputs import get_indicator_series_from_who_data, get_multicountry_df_from_who_data, get_hosp_series_from_owid_data
from emu_renewal.process import CosineMultiCurve
from emu_renewal.distributions import GammaDens
from emu_renewal.renew import MultiStrainModel
from emu_renewal.outputs import get_spaghetti, get_quant_df_from_spaghetti, get_spagh_df_from_dict, add_recovered_to_spaghetti
from emu_renewal.outputs import get_combined_df, get_df_from_3darray
from emu_renewal.plotting import plot_post_prior_comparison, plot_spaghetti_calib_comparison, plot_proc_comparison, plot_mean_proc_diff
from emu_renewal.calibration import StandardCalib
from emu_renewal.targets import StandardDispTarget

In [ ]:
PROJECT_PATH = Path.cwd().resolve()
DATA_PATH = PROJECT_PATH.parent / "data"
AUST_DATA_PATH = PROJECT_PATH.parent / "data/covid_aus"

In [ ]:
# Seroprevalence data
seroprev_data = pd.read_csv(AUST_DATA_PATH / "aus_seroprev_data.csv")
seroprev_data.index = pd.to_datetime(seroprev_data["date"])
aust_seroprev = seroprev_data["seroprevalence"]

# Hospitalisation data
aust_hosp = get_hosp_series_from_owid_data("Daily hospital occupancy", "Australia")

# Variant data
aust_vars = pd.read_csv(AUST_DATA_PATH / "var.csv", index_col=0)
aust_vars.index = pd.to_datetime(aust_vars.index)

# Mobility
aust_mob = pd.read_csv(AUST_DATA_PATH / "aust_mobility.csv", index_col=0)
aust_mob.index = pd.to_datetime(aust_mob.index)
non_resi_mob = aust_mob.loc[:, aust_mob.columns!="residential"]
model_mob = non_resi_mob.mean(axis=1).rolling(7).mean().dropna()

In [ ]:
countries = ["Australia", "United Kingdom of Great Britain and Northern Ireland"]
cases_data = get_multicountry_df_from_who_data("New_cases", countries)
deaths_data = get_multicountry_df_from_who_data("New_deaths", countries)

In [ ]:
# Specify fixed parameters and get calibration data
proc_update_freq = 14
init_time = 50
pop = 26e6
analysis_start = datetime(2021, 12, 1)
analysis_end = datetime(2022, 10, 1)
# Start calibration targets slightly late so as not to penalise laggy indicators
data_start = analysis_start + timedelta(14)
init_start = analysis_start - timedelta(init_time)
init_end = analysis_start - timedelta(1)
select_data = cases_data["Australia"].loc[data_start: analysis_end]
select_deaths = deaths_data["Australia"].loc[data_start: analysis_end]
select_vars = aust_vars.loc[data_start: analysis_end]
hosp_data = aust_hosp[data_start: analysis_end: 7]
init_data = cases_data["Australia"].resample("D").asfreq().interpolate().loc[init_start: init_end] / 7.0

In [ ]:
# Define model and fitter
ba2_seed_times = [datetime(2021, 12, 2), datetime(2021, 12, 10)]
ba5_seed_times = [datetime(2022, 2, 15), datetime(2022, 2, 16)]
seed_times = [ba2_seed_times, ba5_seed_times]
proc_fitter = CosineMultiCurve()
strains = ["ba1", "ba2", "ba5"]
model_args = [
    pop, 
    analysis_start, 
    analysis_end, 
    proc_update_freq, 
    proc_fitter, 
    GammaDens(), 
    init_time, 
    init_data, 
    GammaDens(), 
    GammaDens(), 
    strains, 
    "ba1", 
    seed_times, 
    20.0, 
]
no_mob_model = MultiStrainModel(*model_args + [None])
mob_model = MultiStrainModel(*model_args + [model_mob])

In [ ]:
# Define parameter ranges
priors = {
    "gen_mean": dist.TruncatedNormal(7.3, 0.5, low=1.0),
    "gen_sd": dist.TruncatedNormal(3.8, 0.5, low=1.0),
    "cdr": dist.Beta(15, 15), #(16,40)
    "ifr": dist.Beta(3, 200),
    "rt_init": dist.Normal(0.0, 0.25),
    "report_mean": dist.TruncatedNormal(8.0, 0.5, low=1.0),
    "report_sd": dist.TruncatedNormal(3.0, 0.5, low=1.0),
    "death_mean": dist.TruncatedNormal(18.0, 0.5, low=1.0),
    "death_sd": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "admit_mean": dist.TruncatedNormal(10.0, 1.5, low=1.0),
    "admit_sd": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "stay_mean": dist.TruncatedNormal(10.0, 1.5, low=1.0),
    "stay_sd": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "har": dist.Beta(5, 200),
    "shared_dispersion": dist.HalfNormal(0.5),
    "cross_immunity": dist.Uniform(0.01,0.99),
}

In [ ]:
# Define calibration and calib data
calib_data = {
    "weekly_cases": StandardDispTarget(select_data, weight=20.0),
    "seropos": StandardDispTarget(aust_seroprev, weight=40.0),
    "weekly_deaths": StandardDispTarget(select_deaths, weight=40.0),
    "occupancy": StandardDispTarget(hosp_data, weight=40.0),
    "prop_ba2": StandardDispTarget(select_vars["ba2"], weight=40.0),
}
no_mob_calib = StandardCalib(no_mob_model, priors, calib_data)
mob_calib = StandardCalib(mob_model, priors, calib_data)

In [ ]:
# Run calibration
no_mob_kernel = infer.NUTS(no_mob_calib.calibration, dense_mass=True, init_strategy=no_mob_calib.custom_init(radius=0.5))
mob_kernel = infer.NUTS(mob_calib.calibration, dense_mass=True, init_strategy=mob_calib.custom_init(radius=0.5))
no_mob_mcmc = infer.MCMC(no_mob_kernel, num_chains=2, num_samples=200, num_warmup=200)
mob_mcmc = infer.MCMC(mob_kernel, num_chains=2, num_samples=200, num_warmup=200)
no_mob_mcmc.run(random.PRNGKey(1))
mob_mcmc.run(random.PRNGKey(1))

In [ ]:
# Grab sample of data from calibrated model outputs
no_mob_idata = az.from_dict(no_mob_mcmc.get_samples(True))
mob_idata = az.from_dict(mob_mcmc.get_samples(True))
no_mob_idata_sampled = az.extract(no_mob_idata, num_samples=20)
mob_idata_sampled = az.extract(mob_idata, num_samples=20)
no_mob_sample_params = esamp.xarray_to_sampleiterator(no_mob_idata_sampled)
mob_sample_params = esamp.xarray_to_sampleiterator(mob_idata_sampled)
no_mob_spaghetti = get_spagh_df_from_dict(get_spaghetti(no_mob_calib, no_mob_sample_params))
mob_spaghetti = get_spagh_df_from_dict(get_spaghetti(mob_calib, mob_sample_params))

In [ ]:
plot_spaghetti_calib_comparison(no_mob_spaghetti, no_mob_calib.targets, list(calib_data.keys()) + ["process"]).update_layout(title="No mobility results")

In [ ]:
plot_spaghetti_calib_comparison(mob_spaghetti, mob_calib.targets, list(calib_data.keys()) + ["process"]).update_layout(title="Mobility results")

In [ ]:
plot_proc_comparison(no_mob_idata, mob_idata, ["without mobility", "with mobility"])

In [ ]:
proc_comp_df = pd.concat([no_mob_spaghetti["process"], mob_spaghetti["process"]], keys=["no_mob", "mob"],axis=1)

In [ ]:
proc_comparison_fig = go.Figure()
for i in no_mob_spaghetti["process"]:
    proc_comparison_fig.add_trace(go.Scatter(x=no_mob_spaghetti["process"].index, y=no_mob_spaghetti["process"][i], line={"color": "orange", "width":0.2}))
for i in mob_spaghetti["process"]:
    proc_comparison_fig.add_trace(go.Scatter(x=mob_spaghetti["process"].index, y=mob_spaghetti["process"][i], line={"color": "blue", "width":0.2}))
proc_comparison_fig.update_layout(showlegend=False, height=500, width=800)

In [ ]:
mob_updates = pd.DataFrame(mob_sample_params.components["proc"], columns=mob_model.epoch.index_to_dti(mob_model.x_proc_vals)).T
no_mob_updates = pd.DataFrame(no_mob_sample_params.components["proc"], columns=no_mob_model.epoch.index_to_dti(no_mob_model.x_proc_vals)).T

In [ ]:
update_comparison_fig = go.Figure()
for i in no_mob_updates.columns:
    update_comparison_fig.add_trace(go.Scatter(x=no_mob_updates.index, y=no_mob_updates[i], mode="markers", marker={"color": "orange", "opacity": 0.2, "size": 10.0}))
for i in mob_updates.columns:
    update_comparison_fig.add_trace(go.Scatter(x=mob_updates.index, y=mob_updates[i], mode="markers", marker={"color": "blue", "opacity": 0.2, "size": 10.0}))
update_comparison_fig.update_layout(showlegend=False, height=500, width=800)

In [ ]:
# Get the variable process and dispersion values, with samples as first axis (for use as index to dataframe later)
mob_proc_vals = np.swapaxes(mob_idata.posterior["proc"].to_numpy(), 0, 1)
mob_disp_vals = np.swapaxes(mob_idata.posterior["dispersion_proc"].to_numpy(), 0, 1)
no_mob_proc_vals = np.swapaxes(no_mob_idata.posterior["proc"].to_numpy(), 0, 1)
no_mob_disp_vals = np.swapaxes(no_mob_idata.posterior["dispersion_proc"].to_numpy(), 0, 1)

# Get dataframes from arrays
mob_proc_df = get_df_from_3darray(mob_proc_vals, [0, 2, 1])
no_mob_proc_df = get_df_from_3darray(no_mob_proc_vals, [0, 2, 1])

# Combine the mobility and no mobility dataframes
proc_comparison_df = get_combined_df(mob_proc_df, no_mob_proc_df, "mob", "no_mob")

# Combine the dispersion values into a dataframe
disp_comparison_df = pd.DataFrame(
    {
        "mob": mob_disp_vals.flatten(),
        "no_mob": no_mob_disp_vals.flatten(),
    }
)

In [ ]:
sns.kdeplot(disp_comparison_df, fill=True)

In [ ]:
sns.kdeplot(proc_comparison_df, fill=True)